In [1]:
import pandas as pd
import numpy as np

In [2]:
ibm = pd.read_csv(r"C:\Users\kapoo\burnsightAI\data\ibm_attrition.csv")
perf = pd.read_csv(r"C:\Users\kapoo\burnsightAI\data\employee_performance.csv")
salary = pd.read_csv(r"C:\Users\kapoo\burnsightAI\data\salary_dataset.csv")

In [3]:
for name, df in {
    "IBM": ibm,
    "Performance": perf,
    "Salary": salary
}.items():

    print(name)
    print(df.shape)
    print(df.info())

IBM
(1470, 35)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel 

In [4]:
ibm = ibm.drop_duplicates()
perf = perf.drop_duplicates()
salary = salary.drop_duplicates()

In [5]:
for df in [ibm, perf, salary]:

    num_cols = df.select_dtypes(
        include=np.number
    ).columns

    cat_cols = df.select_dtypes(
        exclude=np.number
    ).columns

    df[num_cols] = df[num_cols].fillna(
        df[num_cols].median()
    )

    for col in cat_cols:
        df[col] = df[col].fillna(
            df[col].mode()[0]
        )

In [6]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

for df in [ibm, perf, salary]:

    for col in df.select_dtypes(
        include="object"
    ).columns:

        le = LabelEncoder()

        df[col] = le.fit_transform(
            df[col]
        )

        encoders[col] = le

In [7]:
ibm["overall_satisfaction"] = (
      ibm["EnvironmentSatisfaction"]
    + ibm["JobSatisfaction"]
    + ibm["RelationshipSatisfaction"]
) / 3

In [8]:
ibm["engagement_score"] = (
      ibm["JobInvolvement"]
    + ibm["WorkLifeBalance"]
) / 2

In [9]:
ibm["salary_growth_ratio"] = (
    ibm["MonthlyIncome"]
    /
    (ibm["TotalWorkingYears"] + 1)
)

In [10]:
ibm["promotion_gap"] = (
    ibm["YearsSinceLastPromotion"]
)

In [11]:
ibm["commute_score"] = (
    ibm["DistanceFromHome"]
)

In [12]:
ibm["BurnoutRisk"] = 0

In [13]:
high = (
    (ibm["WorkLifeBalance"] <= 2)
    &
    (ibm["JobSatisfaction"] <= 2)
    &
    (ibm["EnvironmentSatisfaction"] <= 2)
    &
    (ibm["OverTime"] == 1)
)

ibm.loc[
    high,
    "BurnoutRisk"
] = 2

In [14]:
medium = (
    (ibm["WorkLifeBalance"] <= 2)
    |
    (ibm["JobSatisfaction"] <= 2)
    |
    (ibm["EnvironmentSatisfaction"] <= 2)
)

ibm.loc[
    medium &
    (ibm["BurnoutRisk"] == 0),
    "BurnoutRisk"
] = 1

In [16]:
ibm.to_csv(
    r"C:\Users\kapoo\burnsightAI\data\cleaned_burnout.csv",
    index=False
)